# 3. Aderência, duplicados e triagem pelo título (A3, A4, A5)

In [ ]:
#@title Preparação (rode esta célula primeiro) { display-mode: "form" }
REPOSITORIO = "https://github.com/Juliaalv/curso_proknow.git"
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/curso_proknowc"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPOSITORIO, "/content/curso_proknowc"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/curso_proknowc"], check=True)
    sys.path.insert(0, "/content/curso_proknowc/src")
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from curso_proknowc import colab

pastas = colab.preparar()
print("Tudo pronto. Pasta do curso:", pastas.raiz)

In [ ]:
#@title Entrada { display-mode: "form" }
#@markdown **exemplo do curso**: começa do resultado já pronto do exemplo (curtailment de eólica e solar).
#@markdown **meus arquivos**: começa do resultado que você salvou no notebook anterior.
FONTE = "exemplo do curso" #@param ["exemplo do curso", "meus arquivos"]
padronizada = colab.carregar(pastas, FONTE, "02_padronizado", notebook="03")
print(f"{len(padronizada)} artigos carregados.")

In [ ]:
#@title Artigos disponíveis (para escolher os artigos-semente) { display-mode: "form" }
display(padronizada[["id", "titulo", "ano", "base_origem"]].head(30))

## Teste de aderência (A3)

Escolha pelo menos 2 artigos cujo título indique que são do seu tema. O teste confere se as
palavras-chave deles estão contempladas pelos termos da sua busca.

In [ ]:
#@title Teste de aderência { display-mode: "form" }
#@markdown Separe os termos de um mesmo eixo com ponto e vírgula. Use * como curinga (ex.: curtail*). Deixe em branco os eixos que não precisar.
EIXO_1 = "Curtailment" #@param {type:"string"}
TERMOS_1 = "curtail*; constrained-off; dispatch-down" #@param {type:"string"}
EIXO_2 = "Fontes renováveis (solar e eólica)" #@param {type:"string"}
TERMOS_2 = "wind power; wind energy; wind farm*; photovoltaic*; solar power; solar energy; variable renewable*" #@param {type:"string"}
EIXO_3 = "Soluções / mitigação" #@param {type:"string"}
TERMOS_3 = "energy storage; battery energy storage system*; BESS; flexibility; demand response; hydrogen production; planned load growth" #@param {type:"string"}
EIXO_4 = "Aspectos econômicos" #@param {type:"string"}
TERMOS_4 = "econom*; cost*; revenue*; electricity market*; techno-economic; compensation; carbon targets" #@param {type:"string"}
eixos = colab.ler_eixos([(EIXO_1, TERMOS_1), (EIXO_2, TERMOS_2), (EIXO_3, TERMOS_3), (EIXO_4, TERMOS_4)])

#@markdown Artigos-semente: ids (ex.: R00001) ou trechos do título, separados por ponto e vírgula.
SEMENTES = "" #@param {type:"string"}
from curso_proknowc.aderencia import sementes_da_tabela, testar_aderencia
escolhas = [s.strip() for s in SEMENTES.split(";") if s.strip()]
if len(escolhas) < 2:
    print("Informe pelo menos 2 artigos-semente (id ou trecho do título) para rodar o teste.")
else:
    tabela_aderencia, sugestoes, mensagem = testar_aderencia(eixos, sementes_da_tabela(padronizada, escolhas))
    display(tabela_aderencia)
    print(mensagem)
    if sugestoes:
        print("Sugestões:", "; ".join(sugestoes))

## Duplicados (A4)

O mesmo artigo costuma vir das duas bases. Os registros repetidos são fundidos, completando um com o outro.

In [ ]:
#@title Remoção de duplicados { display-mode: "form" }
#@markdown Semelhança mínima entre títulos para considerar duplicado (os artigos também precisam ter ano e primeiro autor compatíveis).
LIMIAR = 90 #@param {type:"slider", min:80, max:100, step:1}
from curso_proknowc.deduplicacao import deduplicar, sobreposicao
sem_duplicados, fundidos = deduplicar(padronizada, LIMIAR)
duplicados = padronizada[padronizada["id"].isin(fundidos["id_removido"])].assign(motivo_exclusao="duplicado")
colab.registrar_etapa(pastas, "03", "Duplicados", len(padronizada), duplicados)
print(f"{len(fundidos)} duplicado(s) fundido(s). Seguem {len(sem_duplicados)} artigos.")
display(fundidos)
print("Sobreposição entre as bases:")
display(sobreposicao(sem_duplicados))

## Triagem pelo título (A5)

Leia cada título e decida se ele está alinhado ao seu tema. **Quem decide é você.**
Cada clique é salvo na hora no seu Google Drive; se a sessão cair, rode de novo e a triagem continua de onde parou.
Com o exemplo do curso, quase todas as decisões já vêm preenchidas.

In [ ]:
#@title Triagem pelo título { display-mode: "form" }
#@markdown Opcional: descreva o tema para ver primeiro os artigos mais parecidos com ele. A ordem não decide nada.
DESCRICAO_DO_TEMA = "" #@param {type:"string"}
from curso_proknowc import triagem
decisoes = colab.arquivo_decisoes(pastas, FONTE)
fila = triagem.ordenar_por_similaridade(sem_duplicados, DESCRICAO_DO_TEMA) if DESCRICAO_DO_TEMA.strip() else sem_duplicados
display(triagem.interface(fila, "titulo", decisoes))

In [ ]:
#@title Aplicar as decisões e salvar (rode depois de terminar a triagem) { display-mode: "form" }
BAIXAR_CSV = False #@param {type:"boolean"}
try:
    alinhados, nao_alinhados = colab.aplicar_triagem(sem_duplicados, "titulo", decisoes)
except ValueError as erro:
    print(erro)
else:
    colab.registrar_etapa(pastas, "03", "Alinhamento pelo título", len(sem_duplicados), nao_alinhados)
    colab.salvar(pastas, alinhados, "03_alinhados_titulo", baixar=BAIXAR_CSV)
    print(f"Alinhados pelo título: {len(alinhados)}. Não alinhados: {len(nao_alinhados)}. Resultado salvo.")